# AI-lerta v2 — Modelos Regionales de Dengue Colombia
## XGBoost independiente por region geografica

| Mejora | Descripcion |
|--------|-------------|
| Modelos por region | Un XGBoost por cada region IGAC de Colombia |
| Fix data leakage | StandardScaler fitteado **solo en train** de cada region |
| Umbral por region | Umbral optimo F2 calibrado independientemente por region |
| Resultados | Tabla comparativa global + por region |

**Referencia modelo base:** XGBoost global F2-test = 0.762 / AUC = 0.873

---
### Como usar este notebook
1. Abre en Google Colab
2. Ejecuta **Entorno de ejecucion -> Ejecutar todo**
3. Los resultados quedan en `/content/results/`


## Celda 1 — GPU y dependencias

In [ ]:
import subprocess
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout[:200] if r.returncode == 0 else 'Sin GPU — OK para XGBoost')
except:
    print('Sin GPU — se usa CPU')

!pip install -q 'xgboost>=2.0' gdown pyarrow
print('Dependencias listas')


## Celda 2 — Imports y configuracion global

In [ ]:
import pandas as pd
import numpy as np
import gdown
import matplotlib.pyplot as plt
import warnings
import joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    fbeta_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report,
    precision_recall_fscore_support
)
import xgboost as xgb
warnings.filterwarnings('ignore')

for d in ['results', 'models', 'data']:
    Path(d).mkdir(exist_ok=True)

ANIOS_TRAIN = list(range(2009, 2017))
ANIOS_VAL   = list(range(2017, 2020))
ANIOS_TEST  = [2022, 2023, 2024]

COLS_CLIMA = [
    'temp_media_c','humedad_pct','precip_mm',
    'temp_lag2','temp_lag3','temp_lag4',
    'precip_lag2','precip_lag3','precip_lag4',
    'humedad_lag1','humedad_lag2','humedad_lag3',
]

FEATURES = [
    'altitud_msnm','cat_altitud_enc',
    'anio','semana_epi','semana_sin','semana_cos','temporada_lluvias',
    'temp_media_c','humedad_pct','precip_mm',
    'grados_dia','temp_optima','temp_letal','temp_inhibicion','indice_idoneidad',
    'precip_acum4','precip_acum8',
    'temp_lag2','temp_lag3','temp_lag4',
    'precip_lag2','precip_lag3','precip_lag4',
    'humedad_lag1','humedad_lag2','humedad_lag3',
    'casos_lag1','casos_lag2','casos_lag3','casos_lag4',
    'casos_ma4','casos_tendencia',
    'poblacion','anio_epidemico','post_epidemia',
]

print(f'Features: {len(FEATURES)}')
print(f'Train: {ANIOS_TRAIN[0]}-{ANIOS_TRAIN[-1]} | Val: {ANIOS_VAL[0]}-{ANIOS_VAL[-1]} | Test: {ANIOS_TEST}')


## Celda 3 — Cargar datos

In [ ]:
gdown.download(
    'https://drive.google.com/uc?id=1S76FGFaw6Sywj-sfOWcl6UqOCsbb1M2i',
    'data/dataset_dengue_completo.csv', quiet=False
)

df = pd.read_csv('data/dataset_dengue_completo.csv')
df['cod_municipio'] = df['cod_municipio'].astype(str).str.zfill(5)
print(f'Dataset: {len(df):,} filas')
print(f'Annios: {sorted(df["anio"].unique())}')
print(f'Columnas: {df.columns.tolist()}')


## Celda 4 — Pipeline de preparacion

Mismo pipeline del notebook base (05): agrega a nivel municipal, construye grilla completa,
calcula canal endemico y crea todas las features.


In [ ]:
# 4.1 Agregar a nivel municipal
df_mun = (
    df.groupby(
        ['cod_municipio','NOM_MPIO','NOM_DPTO',
         'anio','semana_epi','altitud_msnm','cat_altitud','poblacion'] + COLS_CLIMA
    ).agg(casos=('casos','sum')).reset_index()
)
df_mun['tasa_x100k'] = (df_mun['casos'] / df_mun['poblacion']) * 100_000

# 4.2 Municipios validos (>=20 semanas con casos en periodo train)
df_train_mun      = df_mun[df_mun['anio'].isin(range(2009, 2017))]
semanas_por_mun   = df_train_mun[df_train_mun['casos'] > 0].groupby('cod_municipio').size()
municipios_validos = semanas_por_mun[semanas_por_mun >= 20].index
print(f'Municipios validos: {len(municipios_validos)}')

# 4.3 Grilla completa municipio x anio x semana
info_mun = (
    df_mun[df_mun['cod_municipio'].isin(municipios_validos)]
    [['cod_municipio','NOM_MPIO','NOM_DPTO','altitud_msnm','cat_altitud']]
    .drop_duplicates('cod_municipio')
)
todos_anios = ANIOS_TRAIN + ANIOS_VAL + ANIOS_TEST

grilla = pd.MultiIndex.from_product(
    [municipios_validos, todos_anios, range(1, 53)],
    names=['cod_municipio','anio','semana_epi']
).to_frame(index=False)

casos_ref = (
    df_mun[df_mun['cod_municipio'].isin(municipios_validos)]
    [['cod_municipio','anio','semana_epi','casos','tasa_x100k']]
    .drop_duplicates(['cod_municipio','anio','semana_epi'])
)
clima_ref = (
    df_mun[df_mun['cod_municipio'].isin(municipios_validos)]
    [['cod_municipio','anio','semana_epi'] + COLS_CLIMA]
    .drop_duplicates(['cod_municipio','anio','semana_epi'])
)
pob_ref = (
    df_mun[df_mun['cod_municipio'].isin(municipios_validos)]
    .groupby(['cod_municipio','anio'])['poblacion'].first().reset_index()
)

df_completo = (
    grilla
    .merge(casos_ref, on=['cod_municipio','anio','semana_epi'], how='left')
    .merge(pob_ref,   on=['cod_municipio','anio'], how='left')
    .merge(clima_ref, on=['cod_municipio','anio','semana_epi'], how='left')
    .merge(info_mun,  on='cod_municipio', how='left')
)
df_completo['casos']      = df_completo['casos'].fillna(0)
df_completo['tasa_x100k'] = df_completo['tasa_x100k'].fillna(0)

df_completo = df_completo.sort_values(['cod_municipio','anio','semana_epi'])
for col in COLS_CLIMA:
    df_completo[col] = (
        df_completo.groupby('cod_municipio')[col]
        .transform(lambda x: x.interpolate().ffill().bfill())
    )
print(f'Grilla completa: {len(df_completo):,} filas')

# 4.4 Canal endemico (P25 / mediana / P75)
df_train_con_casos = df_completo[
    df_completo['anio'].isin(ANIOS_TRAIN) & (df_completo['casos'] > 0)
].copy()

canal_mun = (
    df_train_con_casos
    .groupby(['cod_municipio','semana_epi'])['casos']
    .agg(
        p25    =lambda x: x.quantile(0.25),
        mediana=lambda x: x.quantile(0.50),
        p75    =lambda x: x.quantile(0.75),
    ).reset_index()
)

def clasificar_zona(row):
    if pd.isna(row['p25']):                          return 'seguridad'
    if row['p75'] == 0 and row['casos'] == 0:        return 'seguridad'
    if row['p25'] == row['p75']:
        if row['casos'] > row['p75']:                return 'epidemica'
        elif row['casos'] == row['p75'] > 0:         return 'alerta'
        elif row['casos'] == 0:                      return 'seguridad'
        else:                                        return 'exito'
    if row['casos'] >= row['p75']:                   return 'epidemica'
    elif row['casos'] >= row['mediana']:             return 'alerta'
    elif row['casos'] >= row['p25']:                 return 'exito'
    else:                                            return 'seguridad'

df_modelo = df_completo.merge(canal_mun, on=['cod_municipio','semana_epi'], how='left')
df_modelo[['p25','mediana','p75']] = df_modelo[['p25','mediana','p75']].fillna(0)
df_modelo['zona_canal'] = df_modelo.apply(clasificar_zona, axis=1)
df_modelo['alerta']     = df_modelo['zona_canal'].map(
    {'epidemica':1,'alerta':1,'exito':0,'seguridad':0}
)

# 4.5 Variables autoregresivas y biologicas
df_modelo = df_modelo.sort_values(['cod_municipio','anio','semana_epi']).copy()

for lag in [1,2,3,4]:
    df_modelo[f'casos_lag{lag}'] = df_modelo.groupby('cod_municipio')['casos'].shift(lag)

df_modelo['casos_ma4'] = (
    df_modelo.groupby('cod_municipio')['casos']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
)
df_modelo['casos_tendencia']   = df_modelo['casos_lag1'] - df_modelo['casos_lag2']
df_modelo['grados_dia']        = (df_modelo['temp_media_c'] - 18).clip(lower=0)
df_modelo['temp_optima']       = ((df_modelo['temp_media_c'] >= 26) & (df_modelo['temp_media_c'] <= 29)).astype(int)
df_modelo['temp_letal']        = (df_modelo['temp_media_c'] > 35).astype(int)
df_modelo['temp_inhibicion']   = (df_modelo['temp_media_c'] < 16).astype(int)
df_modelo['precip_acum4']      = (
    df_modelo.groupby('cod_municipio')['precip_mm']
    .transform(lambda x: x.rolling(4, min_periods=1).sum())
)
df_modelo['precip_acum8']      = (
    df_modelo.groupby('cod_municipio')['precip_mm']
    .transform(lambda x: x.rolling(8, min_periods=1).sum())
)
df_modelo['indice_idoneidad']  = (
    df_modelo['temp_media_c'] *
    (1 / (1 + np.exp(0.003 * (df_modelo['altitud_msnm'] - 2200))))
)
df_modelo['semana_sin']        = np.sin(2 * np.pi * df_modelo['semana_epi'] / 52)
df_modelo['semana_cos']        = np.cos(2 * np.pi * df_modelo['semana_epi'] / 52)
df_modelo['temporada_lluvias'] = df_modelo['semana_epi'].apply(
    lambda s: 1 if (13 <= s <= 22) or (35 <= s <= 44) else 0
)
df_modelo['anio_epidemico']    = df_modelo['anio'].isin([2010,2013,2016,2019]).astype(int)
df_modelo['post_epidemia']     = df_modelo['anio'].isin([2011,2014,2017]).astype(int)

le = LabelEncoder()
df_modelo['cat_altitud_enc']   = le.fit_transform(df_modelo['cat_altitud'].fillna('Sin dato'))

print(f'Dataset con features: {df_modelo.shape}')
print(f'% alerta global: {df_modelo["alerta"].mean()*100:.1f}%')
print('Pipeline completado')


## Celda 5 — Asignar regiones geograficas IGAC

Cada municipio se asigna a una region segun los primeros 2 digitos de su codigo DANE.

| Region | Dinamica epidemiologica |
|--------|-------------------------|
| Caribe | Transmision casi continua todo el anio, brotes amplios |
| Andina | Picos marcados en temporadas de lluvia (sem 13-22 y 35-44) |
| Pacifico | Alta humedad, brotes focalizados |
| Orinoquia | Patron ligado a ciclos de inundacion |
| Amazonia | Baja densidad poblacional, subregistro posible |


In [ ]:
REGION_MAP = {
    # Caribe
    '08':'Caribe','13':'Caribe','20':'Caribe','23':'Caribe',
    '44':'Caribe','47':'Caribe','70':'Caribe','88':'Caribe',
    # Andina
    '05':'Andina','15':'Andina','17':'Andina','25':'Andina',
    '41':'Andina','54':'Andina','63':'Andina','66':'Andina',
    '68':'Andina','73':'Andina',
    # Pacifico
    '19':'Pacifico','27':'Pacifico','52':'Pacifico','76':'Pacifico',
    # Orinoquia
    '81':'Orinoquia','85':'Orinoquia','50':'Orinoquia','99':'Orinoquia',
    # Amazonia
    '91':'Amazonia','18':'Amazonia','94':'Amazonia','95':'Amazonia',
    '86':'Amazonia','97':'Amazonia',
}

df_modelo['cod_dpto'] = df_modelo['cod_municipio'].str[:2]
df_modelo['region']   = df_modelo['cod_dpto'].map(REGION_MAP).fillna('Andina')

resumen = (
    df_modelo.groupby('region')
    .agg(
        municipios=('cod_municipio','nunique'),
        filas=('alerta','count'),
        pct_alerta=('alerta', lambda x: round(x.mean()*100, 1))
    ).sort_values('municipios', ascending=False)
)
print('Distribucion por region:')
print(resumen.to_string())
print(f'Regiones: {sorted(df_modelo["region"].unique())}')


## Celda 6 — Entrenamiento de modelos regionales

Para cada region:
1. Separa el subconjunto de datos de esa region
2. Fittea `StandardScaler` **solo en train** -> aplica a val y test (fix del data leakage)
3. Ajusta `scale_pos_weight` segun el desbalance real de esa region
4. Busca umbral optimo en validation maximizando F2
5. Guarda modelo en `models/xgb_{region}.json`


In [ ]:
REGIONES = sorted(df_modelo['region'].unique())

XGB_PARAMS_BASE = dict(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = 'logloss',
    random_state     = 42,
    n_jobs           = -1,
    verbosity        = 0,
)

resultados_val = {}
modelos        = {}
scalers        = {}
umbrales       = {}

print('=' * 70)
print(f'{"Region":<12} {"Mun":>5} {"Train":>9} {"Val":>8} {"% Alerta":>9} {"SPW":>5}')
print('-' * 70)

for region in REGIONES:
    df_r = df_modelo[df_modelo['region'] == region].copy()

    tr  = df_r[df_r['anio'].isin(ANIOS_TRAIN)].dropna(subset=FEATURES+['alerta'])
    val = df_r[df_r['anio'].isin(ANIOS_VAL  )].dropna(subset=FEATURES+['alerta'])
    te  = df_r[df_r['anio'].isin(ANIOS_TEST )].dropna(subset=FEATURES+['alerta'])

    if len(tr) < 500 or val['alerta'].sum() < 10:
        print(f'{region:<12}  SKIP - datos insuficientes')
        continue

    X_tr, y_tr   = tr[FEATURES],  tr['alerta'].astype(int)
    X_val, y_val = val[FEATURES], val['alerta'].astype(int)
    X_te, y_te   = te[FEATURES],  te['alerta'].astype(int)

    # FIX DATA LEAKAGE: scaler fitteado SOLO en train
    scaler = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    X_te_sc  = scaler.transform(X_te)
    scalers[region] = scaler

    # scale_pos_weight ajustado por desbalance regional
    spw = max(1, int((y_tr == 0).sum() / (y_tr == 1).sum()))
    params = {**XGB_PARAMS_BASE, 'scale_pos_weight': spw}

    modelo = xgb.XGBClassifier(**params)
    modelo.fit(
        X_tr_sc, y_tr,
        eval_set=[(X_val_sc, y_val)],
        verbose=False
    )

    # Umbral optimo en validation (maximiza F2)
    probs_val = modelo.predict_proba(X_val_sc)[:, 1]
    mejor_f2, mejor_u = 0, 0.5
    for u in np.arange(0.10, 0.90, 0.05):
        yp = (probs_val >= u).astype(int)
        if yp.sum() == 0: continue
        f2 = fbeta_score(y_val, yp, beta=2, zero_division=0)
        if f2 > mejor_f2:
            mejor_f2, mejor_u = f2, round(u, 2)

    umbrales[region] = mejor_u
    modelos[region]  = modelo

    y_pred_val = (probs_val >= mejor_u).astype(int)
    resultados_val[region] = {
        'municipios': df_r['cod_municipio'].nunique(),
        'n_train'   : len(y_tr),
        'n_val'     : len(y_val),
        'n_test'    : len(y_te),
        'pct_alerta': round(y_tr.mean()*100, 1),
        'umbral'    : mejor_u,
        'spw'       : spw,
        'val_f2'    : round(fbeta_score(y_val, y_pred_val, beta=2, zero_division=0), 3),
        'val_auc'   : round(roc_auc_score(y_val, probs_val), 3),
        'val_recall': round(recall_score(y_val, y_pred_val, zero_division=0), 3),
        'val_prec'  : round(precision_score(y_val, y_pred_val, zero_division=0), 3),
        '_X_te': X_te_sc, '_y_te': y_te,
    }

    modelo.save_model(f'models/xgb_{region}.json')
    joblib.dump(scaler, f'models/scaler_{region}.pkl')

    print(
        f'{region:<12} {df_r["cod_municipio"].nunique():>5}'
        f' {len(y_tr):>9,} {len(y_val):>8,}'
        f' {y_tr.mean()*100:>8.1f}% {spw:>5}'
    )

print('=' * 70)
print(f'Modelos entrenados: {len(modelos)} regiones -> guardados en models/')


## Celda 7 — Resultados en Validation (2017-2019)

In [ ]:
df_val_res = pd.DataFrame({
    r: {k: v for k, v in d.items() if not k.startswith('_')}
    for r, d in resultados_val.items()
}).T.sort_values('val_f2', ascending=False)

print('RESULTADOS VALIDATION por REGION')
print('=' * 75)
print(df_val_res[['municipios','n_train','pct_alerta','umbral',
                   'val_f2','val_auc','val_recall','val_prec']].to_string())
print('=' * 75)
print(f'Promedio F2 validation : {df_val_res["val_f2"].astype(float).mean():.3f}')
print('Referencia modelo base : 0.678 (XGBoost global sin regiones)')


## Celda 8 — Evaluacion final en TEST (2022-2024)

> **Ejecutar solo una vez.** Estos son los resultados finales.


In [ ]:
resultados_test = {}

for region, modelo in modelos.items():
    d      = resultados_val[region]
    X_te   = d['_X_te']
    y_te   = d['_y_te']
    umbral = umbrales[region]

    probs_te = modelo.predict_proba(X_te)[:, 1]
    y_pred   = (probs_te >= umbral).astype(int)

    resultados_test[region] = {
        'municipios' : d['municipios'],
        'n_test'     : d['n_test'],
        'umbral'     : umbral,
        'test_f2'    : round(fbeta_score(y_te, y_pred, beta=2, zero_division=0), 3),
        'test_auc'   : round(roc_auc_score(y_te, probs_te), 3),
        'test_recall': round(recall_score(y_te, y_pred, zero_division=0), 3),
        'test_prec'  : round(precision_score(y_te, y_pred, zero_division=0), 3),
        '_probs'  : probs_te,
        '_labels' : y_te.values,
        '_preds'  : y_pred,
    }

df_test_res = pd.DataFrame({
    r: {k: v for k, v in d.items() if not k.startswith('_')}
    for r, d in resultados_test.items()
}).T.sort_values('test_f2', ascending=False)

print('EVALUACION FINAL EN TEST por REGION')
print('=' * 70)
print(df_test_res[['municipios','n_test','umbral',
                    'test_f2','test_auc','test_recall','test_prec']].to_string())
print('=' * 70)

f2_prom  = df_test_res['test_f2'].astype(float).mean()
auc_prom = df_test_res['test_auc'].astype(float).mean()
print(f'COMPARACION GLOBAL')
print(f'{"Modelo":<35} {"F2-Test":>8} {"AUC-Test":>10}')
print('-' * 55)
print(f'{"XGBoost global (base)":<35} {"0.762":>8} {"0.873":>10}')
print(f'{"XGBoost regional v2 (promedio)":<35} {f2_prom:>8.3f} {auc_prom:>10.3f}')

df_test_res.to_csv('results/resultados_test_regional.csv')
print('Guardado: results/resultados_test_regional.csv')


## Celda 9 — Visualizaciones comparativas

In [ ]:
regiones_ok = list(resultados_test.keys())
n    = len(regiones_ok)
cols = 3
rows = int(np.ceil(n / cols))

# Figura 1: Matrices de confusion por region
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
fig.suptitle('Matrices de Confusion por Region - Test 2022-2024',
             fontsize=14, fontweight='bold', y=1.01)
axes = axes.flatten() if n > 1 else [axes]

for i, region in enumerate(regiones_ok):
    d  = resultados_test[region]
    cm = confusion_matrix(d['_labels'], d['_preds'])
    ConfusionMatrixDisplay(cm, display_labels=['NORMAL','ALERTA']).plot(
        ax=axes[i], colorbar=False, cmap='Oranges')
    axes[i].set_title(
        f'{region}\nF2={d["test_f2"]} | AUC={d["test_auc"]}',
        fontsize=10
    )

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('results/confusion_matrices_regional.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: results/confusion_matrices_regional.png')

# Figura 2: F2 y AUC por region
fig2, ax2 = plt.subplots(figsize=(10, 5))
x    = np.arange(len(regiones_ok))
w    = 0.35
f2s  = [float(resultados_test[r]['test_f2'])  for r in regiones_ok]
aucs = [float(resultados_test[r]['test_auc']) for r in regiones_ok]

ax2.bar(x - w/2, f2s,  w, label='F2-score', color='coral',     alpha=0.85)
ax2.bar(x + w/2, aucs, w, label='AUC-ROC',  color='steelblue', alpha=0.85)
ax2.axhline(0.762, color='coral',     linestyle='--', alpha=0.5, label='F2 base global (0.762)')
ax2.axhline(0.873, color='steelblue', linestyle='--', alpha=0.5, label='AUC base global (0.873)')
ax2.axhline(0.70,  color='black',     linestyle=':',  alpha=0.4, label='Meta minima F2=0.70')
ax2.set_xticks(x)
ax2.set_xticklabels(regiones_ok, rotation=15)
ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Score')
ax2.set_title('F2 y AUC por Region - Test 2022-2024', fontweight='bold')
ax2.legend(loc='lower right', fontsize=9)

for rect, val in zip(ax2.patches[:len(regiones_ok)], f2s):
    ax2.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('results/f2_auc_por_region.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: results/f2_auc_por_region.png')

# Figura 3: Curvas ROC por region
fig3, ax3 = plt.subplots(figsize=(8, 6))
colores = plt.cm.tab10(np.linspace(0, 1, len(regiones_ok)))

for i, region in enumerate(regiones_ok):
    d = resultados_test[region]
    RocCurveDisplay.from_predictions(
        d['_labels'], d['_probs'], ax=ax3,
        name=f'{region} (AUC={d["test_auc"]})',
        color=colores[i]
    )

ax3.plot([0,1],[0,1],'k--', alpha=0.4)
ax3.set_title('Curvas ROC por Region - Test 2022-2024', fontweight='bold')
ax3.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('results/roc_curves_regional.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: results/roc_curves_regional.png')


## Celda 10 — Feature importance por region

In [ ]:
cols_fi = 3
rows_fi = int(np.ceil(len(regiones_ok) / cols_fi))

fig, axes = plt.subplots(rows_fi, cols_fi, figsize=(18, 5*rows_fi))
fig.suptitle('Top 15 Features por Region - XGBoost Regional v2',
             fontsize=13, fontweight='bold')
axes = axes.flatten()

for i, region in enumerate(regiones_ok):
    modelo = modelos[region]
    imp    = pd.Series(modelo.feature_importances_, index=FEATURES)
    top15  = imp.nlargest(15)
    axes[i].barh(range(len(top15)), top15.values[::-1], color='steelblue', alpha=0.8)
    axes[i].set_yticks(range(len(top15)))
    axes[i].set_yticklabels(top15.index[::-1], fontsize=8)
    axes[i].set_title(
        f'{region} | F2={resultados_test[region]["test_f2"]} AUC={resultados_test[region]["test_auc"]}',
        fontsize=9
    )
    axes[i].set_xlabel('Importancia')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('results/feature_importance_regional.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: results/feature_importance_regional.png')


## Celda 11 — Resumen final

In [ ]:
print('=' * 65)
print('RESUMEN FINAL - AI-lerta v2 Modelos Regionales')
print('=' * 65)

df_final = pd.DataFrame({
    r: {
        'F2 Val'     : resultados_val[r]['val_f2'],
        'F2 Test'    : resultados_test[r]['test_f2'],
        'AUC Test'   : resultados_test[r]['test_auc'],
        'Recall Test': resultados_test[r]['test_recall'],
        'Mun'        : resultados_test[r]['municipios'],
        'Umbral'     : resultados_test[r]['umbral'],
    }
    for r in regiones_ok
}).T.sort_values('F2 Test', ascending=False)

print(df_final.to_string())
print('-' * 65)
f2m  = df_final['F2 Test'].astype(float).mean()
aucm = df_final['AUC Test'].astype(float).mean()
print(f'Promedio F2  test : {f2m:.3f}  (base global: 0.762)')
print(f'Promedio AUC test : {aucm:.3f}  (base global: 0.873)')
sobre_meta = (df_final['F2 Test'].astype(float) >= 0.70).sum()
print(f'Regiones F2 >= 0.70 : {sobre_meta}/{len(df_final)}')
print()
print('ARCHIVOS GENERADOS')
print('models/  -> xgb_{region}.json, scaler_{region}.pkl')
print('results/ -> resultados_test_regional.csv')
print('results/ -> confusion_matrices_regional.png')
print('results/ -> f2_auc_por_region.png')
print('results/ -> roc_curves_regional.png')
print('results/ -> feature_importance_regional.png')
print()
print('PROXIMOS PASOS')
print('1. Agregar variables ONI/ENSO (ver notebook base)')
print('2. Probar BiLSTM con atencion temporal por region')
print('3. Ensemble: promediar probabilidades XGBoost regional + BiLSTM')
print('4. Calibracion de probabilidades con CalibratedClassifierCV')
